# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}\n")
print(f"Citation (@id): {[citation['@id'] for citation in metadata.citation]}\n")
print(f"Version: {getattr(metadata, 'version', None)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets organize records into RecordSets (each with `@id`). Let's enumerate all RecordSets and their Fields.

In [ ]:
# Find all RecordSets in the dataset
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet

if not record_sets:
    print("No explicit RecordSets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            field_ids = []
            if isinstance(rs['field'], list):
                field_ids = [f['@id'] for f in rs['field']]
            elif isinstance(rs['field'], dict):
                field_ids = [rs['field']['@id']]
            print(f"Fields: {field_ids}")
        else:
            print("Fields: None")

# If no recordSets are present, alternative way (explore dataset interface)
if not record_sets:
    # Using mlcroissant, list all available record_sets detected
    detected_record_sets = dataset.record_sets()
    print("Detected record sets:")
    for rs_id in detected_record_sets:
        print(f"- @id: {rs_id}")
        # Print example record fields
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            print(f"Example record keys: {list(records[0].keys())}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this dataset, RecordSets may not be specified directly in the schema. We'll use the mlcroissant autodetected record sets.

In [ ]:
# Extract data from each record set
record_sets_ids = dataset.record_sets()  # returns a list of @id strings
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet @id: {record_set_id} -> Columns: {df.columns.tolist()}")

# For further analysis, select the primary record set (usually the largest or most relevant one)
main_record_set_id = None
if dataframes:
    main_record_set_id = max(dataframes.items(), key=lambda kv: kv[1].shape[0])[0]
    print(f"\nPrimary RecordSet selected: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, removing outliers, transforming data distributions, or grouping by key attributes.

In [ ]:
# EDA on the main record set
df = dataframes[main_record_set_id].copy()

# List all columns to identify numeric fields
print("Available columns:", df.columns.tolist())

# Let's find a numeric column (e.g., GAD-7 score, PHQ-9 score, PSQ, Age)
candidate_numeric_fields = [c for c in df.columns if (df[c].dtype in ['int64','float64'] or pd.api.types.is_numeric_dtype(df[c])) and not df[c].isnull().all()]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Chosen numeric field for filtering: {numeric_field}")
else:
    numeric_field = None
    print("No numeric fields available for analysis.")

if numeric_field:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic column if present (e.g., 'gender', 'level_of_education', etc.)
    candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and not df[c].isnull().all() and c != numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("Skipping EDA as no numeric field is present in the data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization - Plotting a histogram and boxplot
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric or group fields available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded and explored survey data on mental health indicators among residents of Kilifi County, Kenya.
- Identified and processed numeric and demographic fields using unique Croissant `@id` references.
- Filtered and normalized selected numeric attributes, performed group averages by demographic attributes.
- Visualized main numeric distributions and demographic splits.

This workflow demonstrates the reusability and AI-readiness enabled by Croissant schema and the `mlcroissant` library.